# Vortex Count

Load in a h5 file with full sweep over I and B. This notebook counts the vortices in the psi magnitude heat maps using skikit image thresholding.

High vortex and low vortex regions must use different thresholds due to different contrasts.
These are hardcoded using a lookup table.
- High contrast - threshold = 0.35
- low contrast - threshold = 0.20

Simply edit the filename to be the h5 file and name with sensible RUN_ID

In [1]:
# ================================================================================================
# IMPORTS
# ================================================================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
from skimage import io, color, measure, morphology
from skimage.filters import threshold_otsu
import h5py
from matplotlib.tri import Triangulation
from tqdm.notebook import tqdm
import pandas as pd
import json
import pickle

# ================================================================================================
# H5 file to count vortices of
# ================================================================================================
filename = 'TDGL_Full_I_Sweep.h5'

✓ Libraries imported


In [2]:
# ================================================================================================
# PARAMETERS
# ================================================================================================
THRESHOLD_HIGH_VORTEX = 0.2    # Threshold for high vortex density 
THRESHOLD_LOW_VORTEX = 0.35    # Threshold for low vortex density
MIN_BLOB_AREA = 1              # Minimum pixels per 'blob' (read "vortex")
MAX_BLOB_AREA = 200            # Maximum pixels per 'blob' - else discounted
OPENING_DISK_SIZE = 1          
CLOSING_DISK_SIZE = 1          
RUN_ID = "trilayer"

# ================================================================================================
# LOOKUP TABLE
# ================================================================================================
THRESHOLD_RANGES = {
    1: {'up': {'lower': -10, 'upper': 30}, 'down': {'lower': -30, 'upper': 10}},
    2: {'up': {'lower': -10, 'upper': 30}, 'down': {'lower': -30, 'upper': 10}},
    3: {'up': {'lower': -5, 'upper': 30}, 'down': {'lower': -30, 'upper': 5}},
    4: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': 0}},
    5: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    6: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    7: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    8: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    9: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    10: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    11: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    12: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    13: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    14: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    15: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    16: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    17: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    18: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    19: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    20: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    21: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    22: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    23: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    24: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    25: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    26: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    27: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    28: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    29: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
    30: {'up': {'lower': 0, 'upper': 30}, 'down': {'lower': -30, 'upper': -5}},
}

ADAPTIVE THRESHOLD CONFIGURATION

Threshold when HIGH vortex density (many vortices): 0.2
Threshold when LOW vortex density (few vortices):  0.35

For 1µA: Use 0.2 everywhere EXCEPT in specific B-field ranges
         In exception ranges: use 0.35

For other currents (>1µA): Threshold determined by B-field ranges from table

Min blob area: 1 pixels
Max blob area: 200 pixels
Run ID: adaptive_high_low_vortex_02_035


In [3]:
# ================================================================================================
# THRESHOLD SELECTION FUNCTION
# ================================================================================================

def get_threshold(B_field_mT, current_uA, sweep_direction):
    
    # Default: high vortex density -> use lower threshold (0.2)
    thresh_low = THRESHOLD_HIGH_VORTEX
    thresh_high = THRESHOLD_LOW_VORTEX
    
    # Check if current is in the lookup table
    current_int = int(round(current_uA))
    
    if current_int not in THRESHOLD_RANGES:
        # Current not in table, use default
        return thresh_low
    
    # Get the range for this current and sweep direction
    ranges = THRESHOLD_RANGES[current_int]
    
    if sweep_direction == 'up':
        lower = ranges['up']['lower']
        upper = ranges['up']['upper']
    elif sweep_direction == 'down':
        lower = ranges['down']['lower']
        upper = ranges['down']['upper']
    else:
        raise ValueError(f"Unknown sweep direction: {sweep_direction}")
    
    # Check if B-field is within exception range
    if lower <= B_field_mT <= upper:
        # B-field is within range -> low vortex density -> use 0.35
        return thresh_high
    else:
        # B-field is outside range -> high vortex density -> use 0.2
        return thresh_low


Testing threshold selection for different currents and B-fields:

Current  B (mT)     Sweep    Expected     Description                                       
------------------------------------------------------------------------------------------
✓ 1     µA 0.0       up      0.35         1µA, up sweep, B=0 (in range -10 to 30)           
✓ 1     µA 50.0      up      0.20         1µA, up sweep, B=50 (outside range)               
✓ 1     µA -20.0     down    0.35         1µA, down sweep, B=-20 (in range -30 to 10)       
✓ 1     µA 20.0      down    0.20         1µA, down sweep, B=20 (outside range)             
✓ 3     µA 0.0       up      0.35         3µA, up sweep, B=0 (in range -5 to 30)            
✓ 3     µA -10.0     up      0.20         3µA, up sweep, B=-10 (outside range)              
✓ 3     µA 0.0       down    0.35         3µA, down sweep, B=0 (in range -30 to 5)          
✗ 3     µA -10.0     down    0.35         3µA, down sweep, B=-10 (outside range)            
✓ 5  

In [4]:
# ================================================================================================
# LOAD FUNCTION
# ================================================================================================
def load_all_sol(filename='all_simulations.h5'):
    """Load all solutions from HDF5 file"""
    all_sol_loaded = {}
    
    with h5py.File(filename, 'r') as f:
        I_values = list(f['metadata'].attrs['I_values'])
        
        # Load mesh
        if 'mesh' in f['metadata']:
            all_sol_loaded['mesh'] = {
                'coordinates': f['metadata/mesh/coordinates'][()],
                'elements': f['metadata/mesh/triangulation'][()]
            }
        
        # build mapping
        print("Building current group mapping...")
        I_to_group = {}
        for key in f.keys():
            if 'I_' in key:
                I_str = key.replace('I_', '').replace('uA', '').replace('μA', '')
                try:
                    I_val = float(I_str)
                    I_to_group[I_val] = key
                except:
                    pass
                    
        for I in I_values:
            if I not in I_to_group:
                print(f"  I={I} not found in file")
                continue
            
            group_name = I_to_group[I]
            all_sol_loaded[I] = {'up': {}, 'down': {}}
            current_group = f[group_name]
            
            if 'up' in current_group:
                for B_str in current_group['up'].keys():
                    B = float(B_str.replace('B_', '').replace('mT', ''))
                    B_group = current_group['up'][B_str]
                    data_dict = {}
                    for dataset_name in B_group.keys():
                        try:
                            data_dict[dataset_name] = B_group[dataset_name][()]
                        except:
                            pass
                    all_sol_loaded[I]['up'][B] = data_dict
            
            if 'down' in current_group:
                for B_str in current_group['down'].keys():
                    B = float(B_str.replace('B_', '').replace('mT', ''))
                    B_group = current_group['down'][B_str]
                    data_dict = {}
                    for dataset_name in B_group.keys():
                        try:
                            data_dict[dataset_name] = B_group[dataset_name][()]
                        except:
                            pass
                    all_sol_loaded[I]['down'][B] = data_dict
    
    return all_sol_loaded, I_values

# ================================================================================================
# STORE SOLUTION IN all_sol
# ================================================================================================
all_sol, I_values = load_all_sol(filename)

Loading H5 file...
Building current group mapping...
Found groups: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0]

✓ Loaded 30 current values


In [5]:
# ================================================================================================
# MESH SETUP
# ================================================================================================

# Setup sweep directions and B values
sweep_dirs = ['up', 'down']
B_values = np.linspace(-150, 150, 61)
B_values_T = B_values / 1000.0  # Convert mT to T

# Get mesh
mesh = all_sol['mesh']
coordinates = mesh['coordinates']
elements = mesh['elements']

# Scale to physical device size
scale = 1.0 / 18.62
coordinates_scaled = coordinates * scale

# Create triangulation object
tri_obj = Triangulation(coordinates_scaled[:, 0], coordinates_scaled[:, 1], elements)

Sweep directions: ['up', 'down']
B values: 61 values from -150.0 to 150.0 mT
Total iterations: 30 × 2 × 61 = 3660


In [6]:
# ================================================================================================
# Vortex Counting Function
# ================================================================================================

def count_vortices(psi_magnitude, threshold, min_area=1, max_area=200):
    
    try:
        # Create temporary plot file
        fig, ax = plt.subplots(figsize=(3, 8))
        tcf = ax.tripcolor(tri_obj, psi_magnitude, cmap='viridis', shading='flat')
        ax.set_aspect('equal')
        ax.axis('off')
        plt.tight_layout()
        
        # Save to temporary PNG
        temp_file = '_temp_vortex.png'
        fig.savefig(temp_file, dpi=100, bbox_inches='tight')
        plt.close(fig)
        
        # Load image and convert to grayscale
        img = io.imread(temp_file)
        if img.ndim == 3 and img.shape[2] == 4:  # RGBA
            img = img[:, :, :3]  # Remove alpha channel
        elif img.ndim == 3 and img.shape[2] > 3:
            img = img[:, :, :3]
        
        img_gray = color.rgb2gray(img)
        
        # Apply threshold
        binary = img_gray < threshold
        
        # Morphology
        if OPENING_DISK_SIZE > 0:
            binary = morphology.opening(binary, morphology.disk(OPENING_DISK_SIZE))
        if CLOSING_DISK_SIZE > 0:
            binary = morphology.closing(binary, morphology.disk(CLOSING_DISK_SIZE))
        
        # Label connected components
        labeled = measure.label(binary)
        
        # Get blob (vortex) properties
        blob_props = measure.regionprops(labeled)
        
        # Filter by size
        count = 0
        for prop in blob_props:
            if min_area <= prop.area <= max_area:
                count += 1
        
        # Clean up temp file
        import os
        try:
            os.remove(temp_file)
        except:
            pass
        
        return count
    
    except Exception as e:
        print(f"Error: {e}")
        return None

✓ Function defined


In [7]:
# ================================================================================================
# MAIN BATCH-PROCESSING LOOP
# ================================================================================================
vortex_count = {}
results_list = []  # For pandas DataFrame
threshold_log = {}  # Track which threshold was used

# Nested loops: Current -> Sweep -> B-field
for I in tqdm(I_values, desc='Currents', position=0):
    vortex_count[I] = {}
    threshold_log[I] = {}
    
    for sweep_dir in tqdm(sweep_dirs, desc=f'I={I}', position=1, leave=False):
        vortex_count[I][sweep_dir] = {}
        threshold_log[I][sweep_dir] = {}
        
        for B_mT, B_T in tqdm(zip(B_values, B_values_T), desc=f'{I}-{sweep_dir}', position=2, leave=False, total=len(B_values)):
            try:
                # Get adaptive threshold for this combination
                # Uses current-dependent lookup from THRESHOLD_RANGES
                threshold = get_threshold(B_mT, I, sweep_dir)
                
                # Get psi_magnitude for this combination
                psi = all_sol[I][sweep_dir][B_mT]['psi_magnitude']
                
                # Count vortices with adaptive threshold
                count = count_vortices(psi, threshold, MIN_BLOB_AREA, MAX_BLOB_AREA)
                
                # Store in nested dict
                vortex_count[I][sweep_dir][B_mT] = count
                threshold_log[I][sweep_dir][B_mT] = threshold
                
                # Store for DataFrame
                results_list.append({
                    'current': I,
                    'sweep': sweep_dir,
                    'b_field_mT': B_mT,
                    'b_field_T': B_T,
                    'threshold_used': threshold,
                    'num_vortices': count
                })
            
            except KeyError:
                threshold = get_threshold(B_mT, I, sweep_dir)
                vortex_count[I][sweep_dir][B_mT] = None
                threshold_log[I][sweep_dir][B_mT] = threshold
                print(f" Missing data: I={I}, {sweep_dir}, B={B_mT}")
            
            except Exception as e:
                threshold = get_threshold(B_mT, I, sweep_dir)
                vortex_count[I][sweep_dir][B_mT] = None
                threshold_log[I][sweep_dir][B_mT] = threshold
                print(f" Error: I={I}, {sweep_dir}, B={B_mT}: {e}")


Starting batch processing with CURRENT-DEPENDENT ADAPTIVE THRESHOLD...



Currents:   0%|          | 0/30 [00:00<?, ?it/s]

I=1:   0%|          | 0/2 [00:00<?, ?it/s]

1-up:   0%|          | 0/61 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ================================================================================================
# CONVERT TO DATAFRAME
# ================================================================================================

df = pd.DataFrame(results_list)

## Save Results with RUN_ID

In [ ]:
# ================================================================================================
# SAVE RESULTS
# ================================================================================================

# Nested dictionary (pickle)
pkl_file = f'vortex_count_dict_{RUN_ID}.pkl'
with open(pkl_file, 'wb') as f:
    pickle.dump(vortex_count, f)

# Threshold log
thresh_file = f'threshold_log_{RUN_ID}.pkl'
with open(thresh_file, 'wb') as f:
    pickle.dump(threshold_log, f)

# Configuration
config = {
    'method': 'tripcolor_adaptive_threshold',
    'run_id': RUN_ID,
    'threshold_in_range': THRESHOLD_IN_RANGE,
    'threshold_out_range': THRESHOLD_OUT_RANGE,
    'b_range_min_T': B_RANGE_MIN,
    'b_range_max_T': B_RANGE_MAX,
    'min_blob_area': MIN_BLOB_AREA,
    'max_blob_area': MAX_BLOB_AREA,
    'opening_disk_size': OPENING_DISK_SIZE,
    'closing_disk_size': CLOSING_DISK_SIZE,
    'total_processed': len(results_list),
    'threshold_rule': {
        'up_sweep': f'If 0.10T <= B <= 0.30T: threshold={THRESHOLD_IN_RANGE}, else: {THRESHOLD_OUT_RANGE}',
        'down_sweep': f'If -0.30T <= B <= -0.10T: threshold={THRESHOLD_IN_RANGE}, else: {THRESHOLD_OUT_RANGE}'
    }
}

config_file = f'vortex_count_config_{RUN_ID}.json'
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)